1. Mô tả các thuật toán thực tế đã chọn
Trong đồ án này, tôi ưu tiên sử dụng các thuật toán dựa trên cấu trúc cây (Tree-based models) vì khả năng xử lý xuất sắc dữ liệu dạng bảng và các mối quan hệ phi tuyến phức tạp.

1.1. XGBoost (Extreme Gradient Boosting)
Nguyên lý: Đây là một thuật toán thuộc họ Gradient Boosting Decision Tree (GBDT). XGBoost xây dựng một chuỗi các cây quyết định tuần tự, trong đó mỗi cây mới được huấn luyện để dự đoán sai số (residuals) của các cây trước đó.

Lý do chọn: * Hiệu năng cao: Có khả năng xử lý song song, giúp rút ngắn thời gian huấn luyện trên tập dữ liệu hàng triệu bản ghi.

Chống Overfitting: Tích hợp sẵn các kỹ thuật hiệu chỉnh (L1, L2 Regularization).

Xử lý dữ liệu nhiễu: Cực kỳ hiệu quả với các biến có giá trị ngoại lai (outliers) như tọa độ hoặc thời gian di chuyển bất thường.
2. Mô tả các thuộc tính và đặc trưng (Final Features)
Để mô hình đạt độ chính xác cao nhất (giảm chỉ số RMSLE), dữ liệu thô đã được biến đổi thành bộ đặc trưng (features) cuối cùng sau đây:

2.1. Nhóm Đặc trưng Thời gian (Temporal Features)
Giúp mô hình bắt kịp quy luật giao thông theo chu kỳ của thành phố New York:

pickup_hour: Giờ đón khách (0-23). Xác định các khung giờ cao điểm và giờ thấp điểm.

day_of_week: Thứ trong tuần (Thứ 2 - Chủ Nhật). Phản ánh sự khác biệt giữa lưu lượng giao thông ngày đi làm và ngày nghỉ.

month: Tháng trong năm (1-12). Ảnh hưởng bởi yếu tố mùa và thời tiết.

2.2. Nhóm Đặc trưng Không gian & Khoảng cách (Spatial Features)
Đây là nhóm biến quan trọng nhất để xác định thời gian di chuyển:

dist_haversine: Khoảng cách đường chim bay giữa hai tọa độ (vĩ độ/kinh độ).

dist_manhattan: Khoảng cách di chuyển theo lưới đường bộ (tổng khoảng cách theo trục ngang và dọc), cực kỳ phù hợp với cấu trúc đường phố bàn cờ của New York.

bearing (Hướng di chuyển): Góc di chuyển giữa điểm đón và điểm trả, giúp mô hình phân biệt hướng đi vào trung tâm hay đi ra ngoại ô.

2.3. Nhóm Đặc trưng Cụm địa lý (Clustering Features)
pickup_cluster / dropoff_cluster: Sử dụng thuật toán K-Means để gom nhóm hàng triệu tọa độ rời rạc thành các cụm khu vực (ví dụ: khu vực Sân bay JFK, khu vực Manhattan, khu vực Brooklyn). Điều này giúp mô hình hiểu được đặc thù giao thông của từng vùng địa lý cụ thể.

2.4. Nhóm Đặc trưng Khác
passenger_count: Số lượng hành khách trên xe.

vendor_id: Mã nhà cung cấp dịch vụ vận tải.

store_and_fwd_flag: Trạng thái lưu trữ và gửi dữ liệu của thiết bị định vị trên xe.


Chạy 2 đoạn bên dưới để tải data từ kaggle

In [3]:
# import kagglehub

# # Download latest version
# path = kagglehub.competition_download('nyc-taxi-trip-duration')

# print("Path to competition files:", path)

In [4]:
# import zipfile
# import os

# os.makedirs("data", exist_ok=True)

# for file in os.listdir():
#     if file.endswith(".zip"):
#         print("Extracting:", file)
#         with zipfile.ZipFile(file, 'r') as zip_ref:
#             zip_ref.extractall("data")

# print("All files extracted!")

In [5]:
import xgboost as xgb
import pandas as pd
import numpy as np
import lightgbm as lgb
import time
from datetime import datetime
from catboost import CatBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

1. Đầu tiên là đọc dữ liệu

In [6]:
#Đọc dữ liệu
df = pd.read_csv('data/train.csv')


2. Chuyển đổi giá trị (Data Transformation)

In [7]:
# Chuyển đổi định dạng datetime
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

# Chuyển đổi biến mục tiêu sang Log-scale để giảm độ lệch (Skewness)
df['trip_duration_log'] = np.log1p(df['trip_duration'])

# Chuyển đổi biến phân loại store_and_fwd_flag sang số (Y=1, N=0)
df['store_and_fwd_flag'] = df['store_and_fwd_flag'].map({'Y': 1, 'N': 0})

3. Trích xuất thuộc tính (Feature Extraction)

In [8]:
# A. Nhóm đặc trưng thời gian
df['hour'] = df['pickup_datetime'].dt.hour
df['day_of_week'] = df['pickup_datetime'].dt.weekday
df['month'] = df['pickup_datetime'].dt.month
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

# B. Nhóm đặc trưng khoảng cách
def haversine_array(lat1, lng1, lat2, lng2):
    # Tính khoảng cách đường chim bay (Haversine)
    lat1, lng1, lat2, lng2 = map(np.radians, (lat1, lng1, lat2, lng2))
    AVG_EARTH_RADIUS = 6371  # km
    lat = lat2 - lat1
    lng = lng2 - lng1
    d = np.sin(lat * 0.5) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(lng * 0.5) ** 2
    return 2 * AVG_EARTH_RADIUS * np.arcsin(np.sqrt(d))

def manhattan_distance(lat1, lng1, lat2, lng2):
    # Tính khoảng cách Manhattan
    a = haversine_array(lat1, lng1, lat1, lng2)
    b = haversine_array(lat1, lng1, lat2, lng1)
    return a + b

df['dist_haversine'] = haversine_array(df['pickup_latitude'], df['pickup_longitude'], 
                                        df['dropoff_latitude'], df['dropoff_longitude'])

df['dist_manhattan'] = manhattan_distance(df['pickup_latitude'], df['pickup_longitude'], 
                                           df['dropoff_latitude'], df['dropoff_longitude'])

# C. Tính hướng di chuyển (Bearing)
def bearing_array(lat1, lng1, lat2, lng2):
    lng_delta_rad = np.radians(lng2 - lng1)
    lat1, lng1, lat2, lng2 = map(np.radians, (lat1, lng1, lat2, lng2))
    y = np.sin(lng_delta_rad) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(lng_delta_rad)
    return np.degrees(np.atan2(y, x))

df['direction'] = bearing_array(df['pickup_latitude'], df['pickup_longitude'], 
                                df['dropoff_latitude'], df['dropoff_longitude'])

4. Gộp bảng / làm sạch (Cleaning & Merging)

In [9]:
# Loại bỏ các Outliers rõ rệt để mô hình không bị nhiễu
# Ví dụ: Chuyến đi > 24h (86400s) hoặc < 1 phút (60s)
df = df[(df['trip_duration'] > 60) & (df['trip_duration'] < 86400)]
# Kiểm tra lại 5 dòng đầu tiên
print(df[['hour', 'dist_manhattan', 'direction', 'trip_duration_log']].head())

   hour  dist_manhattan   direction  trip_duration_log
0    17        1.735433   99.970196           6.122493
1     0        2.430506 -117.153768           6.498282
2    11        8.203575 -159.680165           7.661527
3    19        1.661331 -172.737700           6.063785
4    13        1.199457  179.473585           6.077642


Chúng ta cần kiểm tra xem mô hình dự đoán thời gian di chuyển hoạt động ra sao trên những chuyến đi mà nó chưa từng thấy bao giờ. Nếu dùng toàn bộ dữ liệu để huấn luyện, chúng ta sẽ không có cách nào biết được mô hình có bị "học thuộc lòng" các điểm nhiễu (outliers) hay không.

##Chiến lược chia dữ liệu (Train-Test Split)
Thông thường, với tập dữ liệu khoảng 1.45 triệu bản ghi như NYC Taxi, chúng ta sẽ chia theo tỷ lệ phổ biến: 80/20.

Tập Huấn luyện (Train set - 80%): Dùng để "dạy" thuật toán XGBoost hoặc LightGBM nhận diện mối quan hệ giữa tọa độ, thời gian và trip_duration.

Tập Kiểm thử (Test/Validation set - 20%): Dùng làm "bài thi" cuối cùng. Chúng ta sẽ lấy kết quả dự đoán từ tập này để tính toán chỉ số RMSLE.

1. Xây dựng & Triển khai các Mô-đun

MÔ-ĐUN 1: XÂY DỰNG MÔ HÌNH

In [10]:
def build_model():
    """
    Khởi tạo mô hình XGBoost với các tham số tối ưu cơ bản.
    """
    model = xgb.XGBRegressor(
        n_estimators=500,
        max_depth=10,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,         # Sử dụng toàn bộ nhân CPU để chạy nhanh hơn
        random_state=42,
        objective='reg:squarederror'
    )
    return model

MÔ-ĐUN 2: HUẤN LUYỆN (TRAINING)

In [11]:
def train_module(model, X_train, y_train):
    """
    Thực hiện huấn luyện mô hình trên tập Train.
    """
    print("Đang bắt đầu quá trình huấn luyện... (có thể mất vài phút)")
    model.fit(X_train, y_train)
    print("Huấn luyện hoàn tất!")
    return model

MÔ-ĐUN 3: KIỂM THỬ (TESTING)

In [12]:
def test_module(model, X_test, y_test):
    """
    Dự đoán trên tập Test và tính toán sai số RMSLE.
    """
    y_pred = model.predict(X_test)
    
    # Vì y đã được log-transform, MSE của log chính là MSLE của giá trị gốc
    rmse_log = np.sqrt(mean_squared_error(y_test, y_pred))
    
    print(f"Kết quả kiểm thử RMSLE: {rmse_log:.4f}")
    return y_pred, rmse_log

2. Triển khai Mô-đun chính (Main)

In [13]:
def main():
    # Tách X, y
    features = ['vendor_id', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 
                'dropoff_longitude', 'dropoff_latitude', 'store_and_fwd_flag',
                'hour', 'day_of_week', 'month', 'dist_manhattan', 'direction']
    
    X = df[features]
    y = df['trip_duration_log']

    # Chia dữ liệu
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Khởi tạo mô hình
    my_model = build_model()

    # Chạy mô-đun huấn luyện
    my_model = train_module(my_model, X_train, y_train)

    # Chạy mô-đun kiểm thử
    predictions, score = test_module(my_model, X_test, y_test)

    print("--- CHƯƠNG TRÌNH KẾT THÚC THÀNH CÔNG ---")

# Chạy chương trình chính
if __name__ == "__main__":
    main()

Đang bắt đầu quá trình huấn luyện... (có thể mất vài phút)
Huấn luyện hoàn tất!
Kết quả kiểm thử RMSLE: 0.3571
--- CHƯƠNG TRÌNH KẾT THÚC THÀNH CÔNG ---


Liệt kê các tham số (parameters) và giá trị (values)

Trong thuật toán XGBoost, việc chọn tham số là sự cân bằng giữa độ chính xác (bias) và khả năng tổng quát hóa (variance).
...

Huấn luyện mô hình:

In [14]:
def build_model():
    """
    Khởi tạo mô hình XGBoost. 
    Lưu ý: Đưa 'eval_metric' vào đây để tránh lỗi ở hàm .fit()
    """
    params = {
        'n_estimators': 500,
        'max_depth': 10,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'objective': 'reg:squarederror',
        'tree_method': 'hist', 
        'random_state': 42,
        'n_jobs': -1,
        'eval_metric': 'rmse'  # Khai báo eval_metric ở đây
    }
    return xgb.XGBRegressor(**params)

def main():
    # 1. Tách đặc trưng (Giữ nguyên như cũ)
    features = ['vendor_id', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 
                'dropoff_longitude', 'dropoff_latitude', 'store_and_fwd_flag',
                'hour', 'day_of_week', 'month', 'dist_manhattan', 'direction']
    
    X = df[features]
    y = df['trip_duration_log']

    # 2. Chia dữ liệu
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # 3. Khởi tạo mô hình
    regressor = build_model()

    # 4. Huấn luyện mô hình
    print("--- Bắt đầu huấn luyện ---")
    # Loại bỏ eval_metric khỏi hàm fit() để tránh lỗi TypeError
    regressor.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)], # Chỉ cần theo dõi trên tập test là đủ
        verbose=50
    )

    # 5. Đánh giá mô hình
    y_pred = regressor.predict(X_test)
    rmse_log = np.sqrt(mean_squared_error(y_test, y_pred))
    
    print(f"\nHuấn luyện thành công!")
    print(f"Kết quả kiểm thử RMSLE: {rmse_log:.4f}")
    print("--- CHƯƠNG TRÌNH KẾT THÚC THÀNH CÔNG ---")

if __name__ == "__main__":
    main()

--- Bắt đầu huấn luyện ---
[0]	validation_0-rmse:0.72643
[50]	validation_0-rmse:0.39500
[100]	validation_0-rmse:0.37235
[150]	validation_0-rmse:0.36689
[200]	validation_0-rmse:0.36373
[250]	validation_0-rmse:0.36177
[300]	validation_0-rmse:0.36012
[350]	validation_0-rmse:0.35925
[400]	validation_0-rmse:0.35847
[450]	validation_0-rmse:0.35758
[499]	validation_0-rmse:0.35708

Huấn luyện thành công!
Kết quả kiểm thử RMSLE: 0.3571
--- CHƯƠNG TRÌNH KẾT THÚC THÀNH CÔNG ---


In [15]:
def evaluate_models(X_train, X_test, y_train, y_test):
    # Danh sách các mô hình đã thay đổi: Random Forest -> CatBoost
    models = {
        "Linear Regression": LinearRegression(),
        "CatBoost": CatBoostRegressor(iterations=500, learning_rate=0.05, depth=10, silent=True, random_seed=42),
        "LightGBM": lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=31, random_state=42),
        "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
    }
    
    results = {}

    for name, model in models.items():
        print(f"--- Đang huấn luyện: {name} ---")
        start_time = time.time()
        
        # Huấn luyện mô hình
        model.fit(X_train, y_train)
        
        # Dự đoán kết quả
        y_pred = model.predict(X_test)
        
        # Tính toán RMSLE (vì mục tiêu y đã được log-transform trong tiền xử lý)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        duration = time.time() - start_time
        
        results[name] = {"RMSLE": rmse, "Time (s)": duration}
        print(f"Hoàn tất {name} | RMSLE: {rmse:.4f} | Thời gian: {duration:.2f}s\n")
    
    return results

# Triển khai trong hàm main
def main_expansion():
    # Sử dụng bộ đặc trưng đã được trích xuất từ notebook
    features = ['vendor_id', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 
                'dropoff_longitude', 'dropoff_latitude', 'store_and_fwd_flag',
                'hour', 'day_of_week', 'month', 'dist_manhattan', 'direction']
    
    X = df[features]
    y = df['trip_duration_log'] # Giá trị mục tiêu đã qua log-transform
    
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Chạy đánh giá các mô hình
    comparison_results = evaluate_models(X_train, X_test, y_train, y_test)
    
    # Hiển thị bảng so sánh kết quả cuối cùng
    df_res = pd.DataFrame(comparison_results).T
    print("--- BẢNG SO SÁNH KẾT QUẢ ---")
    # Sắp xếp theo chỉ số RMSLE tăng dần (càng thấp càng tốt)
    print(df_res.sort_values(by="RMSLE"))

if __name__ == "__main__":
    main_expansion()

--- Đang huấn luyện: Linear Regression ---
Hoàn tất Linear Regression | RMSLE: 0.5895 | Thời gian: 0.42s

--- Đang huấn luyện: CatBoost ---
Hoàn tất CatBoost | RMSLE: 0.3683 | Thời gian: 67.15s

--- Đang huấn luyện: LightGBM ---
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.031255 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1580
[LightGBM] [Info] Number of data points in the train set: 1159890, number of used features: 12
[LightGBM] [Info] Start training from score 6.487758
Hoàn tất LightGBM | RMSLE: 0.3706 | Thời gian: 13.34s

--- Đang huấn luyện: Gradient Boosting ---
Hoàn tất Gradient Boosting | RMSLE: 0.3866 | Thời gian: 592.03s

--- BẢNG SO SÁNH KẾT QUẢ ---
                      RMSLE    Time (s)
CatBoost           0.368334   67.145457
LightGBM           0.370555   13.338013
Gradient Boosting  0.386643  592.029312
Linear Regr

Sau khi chạy 5 mô hình thì ta thấy XGBoost cho kết quả taháp nhất -> tốt nhất

### Ta sẽ thêm các thuộc tính mới như JFK, LGA, EWR để tính khoảng cách tới sân bay. Lấy các thuộc tính đó bởi vì ở New York, các chuyển taxi đi/đến xân bay là các trường hợp đặc biệt. Chúng thường chạy theo lộ trình cao tốc dài, ít bị dừng đèn đỏ liên tục như trong thành phố, và thường có mức giá/thời gian di chuyển ổn định hơn. 
### Cách làm: ta sẽ tính khoảng cách từ điểm đón (pickup) và điểm trả (dropoff) tới 3 sân bay lớn nhất. Nếu khoảng cách này gần bằng 0, mô hình sẽ hiểu đây là chuyến đi sân bay, quy luật thời gian sẽ khác.
### Thêm biến giờ cao điểm. Vì khi 5km lúc 2 giờ sáng khác hoàn toàn 5km lúc 5 giờ chiều.
### Cách làm: Tạo một biến nhị phân (0 hoặc 1). Nó sẽ bằng 1 nếu chuyến đi rơi vào khung giờ sáng (7h-9h) hoặc chiều (16h-19h) của các ngày trong tuần. Điều này giúp mô hình "cộng thêm" một lượng thời gian đáng kể cho những chuyến đi rơi vào khung giờ kẹt xe này.

### Cùng với việc thêm thuộc tính mới, ta cũng sẽ thêm các kỹ năng mới chẳng hạn như dừng sớm (Early Stopping), lấy mẫu đặc trưng(Subsampling & Colsample) và Giảm Tốc độ học(Learning Rate)

### Dừng sớm: Thay vì để mô hình chạy cố định 2000 cây (n_estimators), chúng ta ra lệnh: "Nếu sau 50 cây liên tiếp mà độ chính xác trên tập kiểm thử không tăng thêm, hãy dừng lại ngay". Lợi ích của việc đó là Giúp mô hình không bị Overfitting (học thuộc lòng dữ liệu cũ mà không dự báo được dữ liệu mới) và tiết kiệm rất nhiều thời gian chạy máy.
### Giảm Tốc độ học: Chúng ta hạ learning_rate từ 0.05 xuống 0.03. Lợi ích của việc đó là khiến cho mô hình "đi chậm lại để quan sát kỹ hơn". Khi đi chậm, mô hình sẽ tìm ra các quy luật nhỏ (subtle patterns) tốt hơn, giúp kết quả cuối cùng mịn và chính xác hơn.
### Lấy mẫu đặc trưng: Sử dụng subsample và colsample_bytree. Mỗi cây quyết định trong XGBoost sẽ không nhìn thấy toàn bộ dữ liệu mà chỉ nhìn thấy một phần ngẫu nhiên (ví dụ 80% dữ liệu và 70% thuộc tính). Kỹ thuật này tạo ra sự đa dạng giữa các cây, giúp mô hình bền bỉ hơn với các dữ liệu nhiễu.


In [ ]:
def add_new_features(data):
    """
    Kỹ thuật đặc trưng: Thêm các biến mới dựa trên kiến thức miền.
    """
    airports = {
        'JFK': (40.6413, -73.7781),
        'LGA': (40.7769, -73.8740),
        'EWR': (40.6895, -74.1745)
    }
    
    for name, (lat, lon) in airports.items():
        data[f'pickup_dist_{name}'] = np.sqrt((data['pickup_latitude'] - lat)**2 + 
                                               (data['pickup_longitude'] - lon)**2)
        data[f'dropoff_dist_{name}'] = np.sqrt((data['dropoff_latitude'] - lat)**2 + 
                                                (data['dropoff_longitude'] - lon)**2)

    data['is_rush_hour'] = ((data['hour'].between(7, 9) | data['hour'].between(16, 19)) & 
                            (data['day_of_week'] < 5)).astype(int)
    return data

def main_improved():
    # Bước 1: Thêm thuộc tính mới
    print("--- Đang tạo thêm đặc trưng mới ---")
    df_enhanced = add_new_features(df.copy())
    
    features = ['vendor_id', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 
                'dropoff_longitude', 'dropoff_latitude', 'store_and_fwd_flag',
                'hour', 'day_of_week', 'month', 'dist_manhattan', 'direction',
                'pickup_dist_JFK', 'dropoff_dist_JFK', 'is_rush_hour']
    
    X = df_enhanced[features]
    y = df_enhanced['trip_duration_log']

    # Bước 2: Chia dữ liệu
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Bước 3: Cấu hình mô hình
    # LƯU Ý: early_stopping_rounds được đưa vào đây
    params = {
        'n_estimators': 2000,
        'max_depth': 8,
        'learning_rate': 0.03,
        'subsample': 0.8,
        'colsample_bytree': 0.7,
        'objective': 'reg:squarederror',
        'tree_method': 'hist', 
        'random_state': 42,
        'n_jobs': -1,
        'eval_metric': 'rmse',
        'early_stopping_rounds': 50  # Đã chuyển về đây
    }
    
    regressor = xgb.XGBRegressor(**params)

    # Bước 4: Huấn luyện
    print("--- Bắt đầu huấn luyện cải tiến ---")
    regressor.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=100
    )

    # Bước 5: Đánh giá
    y_pred = regressor.predict(X_test)
    rmse_log = np.sqrt(mean_squared_error(y_test, y_pred))
    
    print(f"\nKết quả sau khi cải tiến RMSLE: {rmse_log:.4f}")
    
    # In ra Feature Importance
    print("\nTop 5 đặc trưng quan trọng nhất:")
    importance = dict(zip(features, regressor.feature_importances_))
    for k in sorted(importance, key=importance.get, reverse=True)[:5]:
        print(f"- {k}: {importance[k]:.4f}")

if __name__ == "__main__":
    main_improved()

--- Đang tạo thêm đặc trưng mới ---
--- Bắt đầu huấn luyện cải tiến ---
[0]	validation_0-rmse:0.73716
[100]	validation_0-rmse:0.39789
[200]	validation_0-rmse:0.37494
[300]	validation_0-rmse:0.36867
[400]	validation_0-rmse:0.36510
[500]	validation_0-rmse:0.36261
[600]	validation_0-rmse:0.36084
[700]	validation_0-rmse:0.35940
[800]	validation_0-rmse:0.35820
[900]	validation_0-rmse:0.35724
[1000]	validation_0-rmse:0.35645
[1100]	validation_0-rmse:0.35584
[1200]	validation_0-rmse:0.35527
[1300]	validation_0-rmse:0.35487
[1400]	validation_0-rmse:0.35440
[1500]	validation_0-rmse:0.35398
[1600]	validation_0-rmse:0.35371
[1700]	validation_0-rmse:0.35346
[1800]	validation_0-rmse:0.35321
[1900]	validation_0-rmse:0.35301
[1999]	validation_0-rmse:0.35276

Kết quả sau khi cải tiến RMSLE: 0.3528

Top 5 đặc trưng quan trọng nhất:
- dist_manhattan: 0.5257
- hour: 0.0613
- direction: 0.0530
- pickup_longitude: 0.0502
- dropoff_dist_JFK: 0.0470
